# EDA & Cleaning: IWPC Subset Cohort (n=1780)

Source: `iwpc_simsl/iwpc_1780_patients.xlsx`, a smaller real-patient IWPC subset bundled with the `simsl` R package. Includes **both INR and weekly dose**, plus genotype dummy variables, making it useful for INR-aware modeling, not just dose.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 100)
DATA_DIR = Path("..").resolve()
DECISIONS_PATH = Path("dataset_decisions.json")

def record_decision(name, verdict, reason, n_rows=None, n_cols=None, saved_to=None):
    decisions = {}
    if DECISIONS_PATH.exists():
        decisions = json.loads(DECISIONS_PATH.read_text())
    decisions[name] = {
        "verdict": verdict,
        "reason": reason,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "saved_to": saved_to,
    }
    DECISIONS_PATH.write_text(json.dumps(decisions, indent=2))
    print(f"Recorded decision for '{name}': {verdict}")


## File inventory: `iwpc_simsl/` (1 data file plus the external `simsl` R package repo)

**Used:** `iwpc_1780_patients.xlsx`.

**Not used:** `simsl/` is the external R package this dataset was delivered alongside, kept as a SHAP-explainability methodology reference. It bundles two R data files: `simsl/data/chicago.RData` (the package's own unrelated demo dataset, not warfarin) and `simsl/data/warfarin.RData` (unverified: `pyreadr` parsed the file but returned zero data.frames, suggesting a non-data.frame R object. Likely redundant with `iwpc_1780_patients.xlsx` shipped in the same delivery, but not confirmed identical. Low priority, not blocking).

In [ ]:
df = pd.read_excel(DATA_DIR / "usable/iwpc_simsl/iwpc_1780_patients.xlsx")
print(df.shape)
df.head(3)

## Missingness & summary stats

In [ ]:
df.isna().mean().sort_values(ascending=False).to_frame("pct_missing")

In [ ]:
df.describe()

## Note on scaling
`Weight` and `Height` appear pre-standardized (z-scored) rather than raw units, and `Age` looks like a transformed or binned value rather than raw years. Any joint modeling with the IWPC-6256 cohort (raw units) will need re-harmonizing these columns. Flagged here, not fixed, since fixing requires knowing the original scaler.

In [ ]:
print(df[["Weight", "Height", "Age"]].describe())

## Cleaning

In [ ]:
clean = df.dropna(subset=["WeeklyDose_mg"])
print(f"Rows before: {len(df)}, after dropping missing dose: {len(clean)}")
out_path = Path("cleaned/iwpc_1780_clean.csv")
clean.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())

## Verdict

**KEEP, secondary/validation modeling dataset.** Real patients, genotype plus clinical covariates plus INR plus dose. Smaller and pre-scaled differently from the 6256 cohort, so use as an independent validation set rather than pooling directly.

In [ ]:
record_decision(
    "iwpc_1780",
    verdict="KEEP",
    reason="Secondary real-patient cohort with genotype, INR, and dose; use for validation (scaling differs from the 6256 cohort).",
    n_rows=clean.shape[0], n_cols=clean.shape[1], saved_to=str(out_path),
)